# 01 · EDA y limpieza (E1)

MineTrac · Máquinas Centella. Solo información **real**: bitácora de reparaciones de los 7 camiones y catálogo de 21 equipos. La operación (horas, toneladas, tarifas, ingresos) está pendiente de entrega por la empresa y no se simula. Ejecutar desde `notebooks/` después de `python scripts/pipeline_e1.py`.

In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'services/tablero')); sys.path.insert(0, str(ROOT / 'services/worker'))
from core import resumen, eventos, serie_diaria, spectral, ciclo_cercano
mant = pd.read_csv(ROOT / 'data/clean/mantenimiento.csv')
inv = pd.read_csv(ROOT / 'data/clean/inventario.csv')
res = json.loads((ROOT / 'data/clean/resultados.json').read_text(encoding='utf-8'))
print(len(inv), 'equipos en el catálogo;', len(mant), 'registros por sistema en la bitácora')

21 equipos en el catálogo; 317 registros por sistema en la bitácora


## 1. Catálogo de equipos

In [2]:
inv.groupby('categoria').size()

categoria
Camión              7
Cargador           11
Minicargador        2
Retroexcavadora     1
dtype: int64

## 2. Los cinco errores de la bitácora real

In [3]:
pd.Series({k: v for k, v in res['auditoria_bitacora'].items() if k != 'formato'}, name='conteo')

intervenciones_leidas        257.0
registros_raw                317.0
registros_clean              317.0
duplicados                     0.0
negativos                      0.0
atipicos                       1.0
limite_atipico_mxn         52900.0
nulos_costo                  182.0
intervenciones               257.0
nulos_fecha_imputados         15.0
proveedores_codificados       13.0
formato_total                 16.0
Name: conteo, dtype: float64

In [4]:
pd.Series(res['auditoria_bitacora']['formato'], name='casos de formato corregidos')

fechas_texto_interpretadas              4
fechas_anio_corregido                   3
fechas_solo_anio                        1
filas_con_orden_alterado_conservadas    3
texto_en_columna_costo                  1
costo_extraido_de_texto                 6
costo_en_columna_servicio               1
Name: casos de formato corregidos, dtype: int64

In [5]:
ev = eventos(mant)
print('Intervenciones:', len(ev), '· con monto:', int(ev.costo_mxn_intervencion.notna().sum()), f'({100*ev.costo_mxn_intervencion.notna().mean():.0f}%)')
# El monto repartido entre sistemas conserva el total de cada intervención.
assert np.isclose(mant.costo_mxn.sum(), ev.costo_mxn_intervencion.sum())
assert not mant.naturaleza_dato.ne('real').any()
ev.fecha_calidad.value_counts()

Intervenciones: 257 · con monto: 75 (29%)


fecha_calidad
ok                        234
imputada_fila_anterior     15
texto_interpretado          4
anio_corregido              3
solo_anio                   1
Name: count, dtype: int64

## 3. Componentes que más gasto concentran (solo montos anotados)

In [6]:
mant.groupby('componente').costo_mxn.sum().sort_values(ascending=False).head(8).round(0)

componente
Cabeza, monobloc y motor mayor       281538.0
Clutch                               152333.0
Cabina y estética                     88617.0
Gato hidráulico y levante             68667.0
Inyectores y bomba de diésel          67615.0
Transmisión, flecha y diferencial     55750.0
Computadora y sensores                33900.0
Compresor                             27000.0
Name: costo_mxn, dtype: float64

## 4. Indicadores y FFT de las intervenciones diarias

In [7]:
pd.Series(resumen(mant))

intervenciones         257.000000
gasto_mxn           928832.000000
con_monto               75.000000
pct_con_monto           29.182879
promedio_mxn         12384.426667
reincidencias           69.000000
pct_reincidencia        26.848249
dtype: float64

In [8]:
serie = serie_diaria(mant, inicio='2024-04-01')
spec, picos = spectral(serie)
print(len(serie), 'días')
print('Ciclo de 7 días:', ciclo_cercano(spec, 7))
print('Ciclo de 15 días:', ciclo_cercano(spec, 15))
pd.DataFrame(picos[:3])

926 días
Ciclo de 7 días: {'periodo_dias': 7, 'amplitud': 0.20889728435286967, 'veces_mediana': 4.561336389543723}
Ciclo de 15 días: {'periodo_dias': 15, 'amplitud': 0.07022239593103055, 'veces_mediana': 1.5333275916602724}


,periodo_dias,amplitud,frecuencia_ciclos_dia
0,7.015152,0.208897,0.142549
1,2.332494,0.171060,0.428726
2,5.202247,0.123863,0.192225


**Lectura.** El ciclo de 7 días destaca; el de 15 días queda al nivel del ruido. Con 257 intervenciones la señal es débil; se repetirá con las horas trabajadas reales cuando la empresa entregue su Excel consolidado.